# 04 · Fault-injection attack loop — crowbar and/or EMFI over the target UART

This is the full **attack → observe → classify** loop against a real target.

Our target is an **Arduino Uno** running `arduino/uart_glitch_target`: send it the command byte and it counts to 10000, then reports the count **back over FaultyCat's own target UART**. FaultyCat voltage-glitches the Uno's 5 V rail mid-count — so if the count that comes back isn't 10000, you know the glitch landed.

We let `fc.GlitchController` do the bookkeeping: it owns the sweep, the per-attempt classification, and the result map, leaving the loop body to handle only the target-specific glitch-and-observe process.

**Wiring**:

* Crowbar **OUTPUT** → Uno **5V**
* FaultyCat **GP0** → Uno **D2**
* Uno **D3** → 1 kΩ → FaultyCat **GP1**
* **Common GND**
* FaultyCat **GP2** → Uno **RESET** for clean-state resets (`RST_GP`)


In [ ]:
import time
import faultycat as fc

# --- edit for your target ---
SIM       = False              # True to dry-run without a board
CMD       = b'\xAA'           # byte that makes the target run + report
EXPECTED  = b'10000'           # the NORMAL answer (count, ASCII) == no fault
BAUD      = 9600               # target UART baud (matches the sketch)

# Engines fired each attempt — ONE loop, both optional. Enable one or BOTH:
USE_CROWBAR = True             # voltage glitch (crowbar on the 5 V rail)
USE_EMFI    = False            # EM glitch (coil over the chip) — HV! True only when set up + safe
OUTPUT        = 'lp'           # crowbar path: 'lp' or 'hp'
EMFI_WIDTH_US = 8              # EMFI pulse width (us), fixed; the swept `width` axis is the crowbar's (ns)
GAP_US        = 100            # inter-pulse gap for both engines (delay_us)

# Two ways to give a sweep vector (set_range accepts any iterable):
WIDTHS_NS = [500, 1000, 2000, 4000, 6000, 8000, 10000, 12000]  # explicit list — crowbar pulse width (ns)
PULSES    = range(1, 9)        # range(start, stop) — 1..8 GLITCH PULSES per trigger (multipulse, both engines)

# Hardware target reset (clean state each try). GP2 -> Arduino RESET. Reliable
# with the patched firmware (LOW->HIGH->hi-Z) + shifter VREF at target Vcc (5 V).
RST_GP    = 2                  # None to skip (passive brownout recovery)
RST_MS    = 10

cat = fc.connect(simulator=SIM)
cat.uart.open(baud=BAUD)
print('target says:', cat.uart.readline(timeout=1.5))   # its BOOT banner
cat.crowbar

## Glitch Result Classification

**In this condition, you define which outcomes are considered successful based on the target's response and observed behavior.**

In our case, `EXPECTED = b'10000'` represents the target's normal behavior: after receiving the command, the Arduino completes the count up to `10000` and returns that value.

Based on this response, we can classify the outcome of each attempt:

* `b'10000'` → **Normal behavior (`EXPECTED`)**. The glitch did not observably alter the execution.
* **Any value other than `b'10000'`** → **Successful glitch (`FAULT`)**. The fault injection altered the target's execution.
* `b'BOOT'` → **Target rebooted (`REBOOT`)**. The glitch caused the target to restart during execution.
* **No response** → **Target hung (`HANG`)**. The target stopped responding during the attempt.

This allows each attempt to be classified according to the response received from the target. Any result that differs from the expected behavior indicates that the glitch had an observable effect on execution; depending on the response, we can determine whether it caused a count alteration, a reboot, or a hang.


In [ ]:
def classify(line: bytes) -> str:
    s = line.strip()
    if s == EXPECTED:  return 'normal'    # target unfazed
    if not s:          return 'crash'     # no answer — hung
    if b'BOOT' in s:   return 'reset'     # banner ANYWHERE => the glitch rebooted the
                                          # target (read_until swallows it when the
                                          # corrupted count lost its '\n' terminator)
    return 'success'                      # corrupted count, target survived — a clean fault

## The Attack Loop — Crowbar and/or EMFI

`GlitchController` is responsible for systematically iterating over the **pulses × width** parameter grid. For each combination, the loop body performs a single fault-injection attempt and records its outcome.

```python
gc = fc.GlitchController(
    ['pulses', 'width'],
    groups=['success', 'reset', 'crash', 'normal', 'error']
)
gc.set_range('pulses', PULSES)
gc.set_range('width', WIDTHS_NS)
```

Here, the two parameters to be swept are defined:

* **`pulses`** determines the number of glitch pulses applied during each attempt.
* **`width`** determines the crowbar pulse width, expressed in nanoseconds.
* The groups defined in `GlitchController` allow the different outcomes obtained during the sweep to be classified and counted: `success`, `reset`, `crash`, `normal`, and `error`.

The enabled injection engines are then configured:

```python
if USE_CROWBAR:
    cat.crowbar.trigger = 'immediate'
    cat.crowbar.output = OUTPUT
    cat.crowbar.delay_us = GAP_US

if USE_EMFI:
    cat.emfi.trigger = 'immediate'
    cat.emfi.delay_us = GAP_US
    cat.emfi.width_us = EMFI_WIDTH_US
```

This allows the **crowbar**, **EMFI**, or both to be used simultaneously. The rest of the attack flow remains unchanged regardless of which injection engine is selected.

During each iteration of `gc.glitch_values()`, the parameters for the current attempt are applied:

```python
for p in gc.glitch_values():
    if USE_CROWBAR:
        cat.crowbar.repeat = p['pulses']
        cat.crowbar.width_ns = p['width']

    if USE_EMFI:
        cat.emfi.repeat = p['pulses']
```

For example, if `pulses = 3`, both engines will be configured to generate three pulses during that attempt.

If `RST_GP` is configured, the target is reset before each attempt so that every test starts from a known state:

```python
if RST_GP is not None:
    cat.target_reset(RST_GP, RST_MS)
    time.sleep(2.0)
```

When EMFI is enabled, its high-voltage capacitor must be charged **before the target's counting window begins**. Otherwise, the time required to charge the capacitor could cause the Arduino's approximately 45 ms window to expire before the glitch is applied:

```python
if USE_EMFI:
    cat.emfi.arm()
    cat.emfi.wait_for_charged()

if USE_CROWBAR:
    cat.crowbar.arm()
```

Once the injection engines are prepared, the UART input buffer is cleared to remove any stale data or boot messages. `CMD` is then sent to the target, which starts its approximately 45 ms counting window:

```python
cat.uart.reset_input()
cat.uart.write(CMD)
time.sleep(0.003)
```

The enabled injection engines are then fired:

```python
if USE_CROWBAR:
    cat.crowbar.fire()

if USE_EMFI:
    cat.emfi.fire()
```

Both mechanisms operate within the same counting window. With this configuration, the crowbar and EMFI are fired consecutively through separate `fire()` calls.

After the attempt, the engines are disarmed and the target's response is read:

```python
if USE_CROWBAR:
    cat.crowbar.disarm()

if USE_EMFI:
    cat.emfi.disarm()

line = cat.uart.read_until(b'\n', timeout=0.3)
res = classify(line)
gc.add(res)
```

The `classify()` function analyzes the received response and determines the outcome of the attempt. `gc.add(res)` then records that classification in `GlitchController`, allowing the results to be counted by category.

The result of each iteration is finally displayed together with the parameters used:

```python
print(
    f"{p['pulses']:>6} {p['width']:>9}  "
    f"{res:<8}  {line!r}"
)
```

This makes it possible to observe the relationship between **injection parameters → target response → result classification** throughout the sweep.

Once the sweep is complete:

```python
gc.counts()
```

reports the number of attempts recorded in each outcome category.

Overall, the flow of each attempt is:

**configure parameters → prepare the target → charge EMFI if required → send `CMD` → apply the glitch → read the response → classify the outcome → record the result → move to the next parameter combination.**


In [ ]:
gc = fc.GlitchController(['pulses', 'width'],
                        groups=['success', 'reset', 'crash', 'normal', 'error'])
gc.set_range('pulses', PULSES)           # range — glitch pulses (both engines)
gc.set_range('width', WIDTHS_NS)         # explicit list — crowbar pulse width (ns)

if USE_CROWBAR:
    cat.crowbar.trigger = 'immediate'; cat.crowbar.output = OUTPUT; cat.crowbar.delay_us = GAP_US
if USE_EMFI:
    cat.emfi.trigger = 'immediate'; cat.emfi.delay_us = GAP_US; cat.emfi.width_us = EMFI_WIDTH_US

fired = ' + '.join(e for e, on in [('crowbar', USE_CROWBAR), ('emfi', USE_EMFI)] if on)
print(f"firing: {fired or 'NOTHING — enable USE_CROWBAR and/or USE_EMFI'}")
print(f"{'pulses':>6} {'width_ns':>9}  outcome   reply")
for p in gc.glitch_values():
    if USE_CROWBAR:
        cat.crowbar.repeat = p['pulses']; cat.crowbar.width_ns = p['width']
    if USE_EMFI:
        cat.emfi.repeat = p['pulses']
    if RST_GP is not None:
        cat.target_reset(RST_GP, RST_MS)   # clean known state each try (their thorough_reset_dut)
        time.sleep(2.0)                    # let the Uno boot back to its idle loop (~1.8 s)
    if USE_EMFI:                           # charge HV BEFORE sending CMD — the multi-second
        cat.emfi.arm(); cat.emfi.wait_for_charged()   # charge would otherwise let the ~45 ms
    if USE_CROWBAR:                        # count finish before we fire (glitch lands too late)
        cat.crowbar.arm()
    cat.uart.reset_input()                 # drop any BOOT banner / stale bytes
    cat.uart.write(CMD)                    # NOW the target starts its ~45 ms count
    time.sleep(0.003)                      # let it enter the loop
    try:
        if USE_CROWBAR:
            cat.crowbar.fire()             # voltage glitch ...
        if USE_EMFI:
            cat.emfi.fire()                # ... then EM pulse (same count window)
        if USE_CROWBAR:
            cat.crowbar.disarm()
        if USE_EMFI:
            cat.emfi.disarm()
    except fc.EngineError:
        gc.add('error'); print(f"{p['pulses']:>6} {p['width']:>9}  error"); continue
    line = cat.uart.read_until(b'\n', timeout=0.3)
    res  = classify(line)
    gc.add(res)
    print(f"{p['pulses']:>6} {p['width']:>9}  {res:<8}  {line!r}")   # per-iteration result
    if RST_GP is None and res in ('crash', 'reset'):
        time.sleep(1.5)             # no reset wire: wait out the brownout reboot

gc.counts()

## Results
This command visualizes the results collected by the GlitchController during the parameter sweep. The X-axis represents the number of glitch pulses (pulses), while the Y-axis represents the crowbar pulse width (width) in nanoseconds. Each point corresponds to a specific combination of injection parameters and is colored according to the classified outcome observed on the target. This provides a visual representation of the parameter space and makes it easier to identify regions associated with different glitch outcomes, including normal execution, resets, crashes, or other observable effects.

In [ ]:
gc.plot(x='pulses', y='width');

## Diagnostic — does each engine actually affect the target?

This is the same trick we used to prove the crowbar (10 µs → brownout → the Uno
reboots): fire **each engine on its own** and watch the target.

If an engine alone produces a non-`normal` outcome — the count corrupts, or a `BOOT`/reset
shows up — that engine is reaching the target.

Run this once to confirm **EMFI works** before you trust a full sweep. If EMFI-only comes back all `normal`, the
coil isn't coupling: reposition it over the chip, or it's simply too weak.

In [ ]:
RUN_DIAG = False   # SAFETY: fires HV (EMFI) — coil over the chip, shield on

DIAG_N          = 5      # attempts per engine
DIAG_PULSES     = 4      # pulses per trigger (both engines)
DIAG_CB_WIDTH   = 8000   # crowbar pulse width (ns)
DIAG_EMFI_WIDTH = 12     # EMFI pulse width (us)

if not RUN_DIAG:
    print("Diagnostic off (RUN_DIAG=False). Set True with the coil placed + shield on.")
else:
    from collections import Counter
    cat.crowbar.trigger = 'immediate'; cat.crowbar.output = OUTPUT; cat.crowbar.delay_us = GAP_US
    cat.crowbar.width_ns = DIAG_CB_WIDTH; cat.crowbar.repeat = DIAG_PULSES
    cat.emfi.trigger = 'immediate'; cat.emfi.delay_us = GAP_US
    cat.emfi.width_us = DIAG_EMFI_WIDTH; cat.emfi.repeat = DIAG_PULSES

    for label, uc, ue in [('EMFI only', False, True), ('crowbar only', True, False), ('both', True, True)]:
        out = Counter()
        for _ in range(DIAG_N):
            if RST_GP is not None:
                cat.target_reset(RST_GP, RST_MS); time.sleep(2.0)   # clean state each try
            if ue: cat.emfi.arm(); cat.emfi.wait_for_charged()  # charge HV before CMD
            if uc: cat.crowbar.arm()
            cat.uart.reset_input(); cat.uart.write(CMD); time.sleep(0.003)
            try:
                if uc: cat.crowbar.fire()                           # voltage ...
                if ue: cat.emfi.fire()                              # ... then EM
                if uc: cat.crowbar.disarm()
                if ue: cat.emfi.disarm()
            except fc.EngineError:
                out['error'] += 1; continue
            out[classify(cat.uart.read_until(b'\n', timeout=0.3))] += 1
        hit = sum(v for k, v in out.items() if k != 'normal')
        print(f"{label:>13}: {dict(out)}   -> {'AFFECTS TARGET' if hit else 'no effect'}")

## Clean up

`cat.uart.close()` explicitly closes the target UART channel, releasing the data port and, if the firmware bridge was opened during the session, shutting it down cleanly. `cat.close()` then releases all remaining resources held by the session, including the EMFI, crowbar, and scanner USB channels, freeing the associated ports so they can be used by another program or by a fresh run of the notebook.

Since `cat.close()` also closes the UART internally, calling `cat.uart.close()` first is not strictly necessary. It simply makes the cleanup sequence explicit by releasing the target communication channel before closing the rest of the session. Both calls are safe to invoke multiple times, and neither raises an exception if the corresponding port is already closed.

In [ ]:
cat.uart.close()
cat.close()